In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sqlalchemy import create_engine, text
from langgraph.graph import END, StateGraph
from typing import TypedDict

In [13]:
DATABASE_URL = "sqlite:///casa_elena_mockup.db"  # Cambia esto según tu configuración
engine = create_engine(DATABASE_URL)

In [40]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from sqlalchemy import create_engine, text
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

# Definir el estado de la consulta
class QueryState(TypedDict):
    user_input: str
    sql_query: str
    is_relevant: bool
    result: str

# Inicializar el modelo de lenguaje
llm = ChatOpenAI(model="ft:gpt-3.5-turbo-0125:growists::B5woYaLN", temperature=0)

# Base de datos SQLite
engine = create_engine("sqlite:///casa_elena_mockup.db")

# Paso 1: Verificar si la consulta es relevante
check_relevance_prompt = PromptTemplate.from_template(
    """
    Dada la siguiente consulta en lenguaje natural, responde "YES" si es relevante para la base de datos, 
    de lo contrario, responde "NO".
    Consulta: {user_input}
    """
)

check_relevance_chain = check_relevance_prompt | llm | StrOutputParser()

def check_relevance(state: QueryState):
    response = check_relevance_chain.invoke({"user_input": state["user_input"]})
    state["is_relevant"] = response.strip().upper() == "YES"
    return state

# Paso 2: Convertir lenguaje natural a SQL
sql_conversion_prompt = PromptTemplate.from_template(
    """
    Convierte la siguiente consulta en lenguaje natural a una consulta SQL válida para SQLite.
    Consulta: {user_input}
    """
)

sql_conversion_chain = sql_conversion_prompt | llm | StrOutputParser()

def convert_to_sql(state: QueryState):
    sql_query = sql_conversion_chain.invoke({"user_input": state["user_input"]})
    state["sql_query"] = sql_query
    return state

# Paso 3: Ejecutar la consulta SQL
def execute_sql(state: QueryState):
    try:
        with engine.connect() as connection:
            result = connection.execute(text(state["sql_query"]))
            state["result"] = str([row for row in result])
    except Exception as e:
        state["result"] = f"Error ejecutando SQL: {str(e)}"
    return state

# Construcción del flujo
graph = StateGraph(QueryState)
graph.add_node("check_relevance", check_relevance)
graph.add_node("convert_to_sql", convert_to_sql)
graph.add_node("execute_sql", execute_sql)

graph.add_conditional_edges(
    "check_relevance",
    lambda state: "convert_to_sql" if state["is_relevant"] else END,
    {"convert_to_sql": "convert_to_sql", END: END}
)
graph.add_edge("convert_to_sql", "execute_sql")
graph.add_edge("execute_sql", END)

graph.set_entry_point("check_relevance")
app = graph.compile()

# Ejemplo de ejecución
if __name__ == "__main__":
    user_input = "¿Cuántos pedidos hay en la base de datos?"
    state = app.invoke({"user_input": user_input})
    print("Resultado final:", state["result"])

Resultado final: Error ejecutando SQL: (sqlite3.OperationalError) no such table: pedidos
[SQL: SELECT COUNT(*) FROM pedidos;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [42]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.sql_database import SQLDatabase
from langchain.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.schema.runnable import RunnablePassthrough
from operator import itemgetter

# Definir el estado de la consulta
class QueryState(TypedDict):
    user_input: str
    sql_query: str
    result: str
    query: str  # Agregar la variable 'query' para la plantilla

# Conexión a la base de datos SQLite usando el directorio de configuración
db = SQLDatabase.from_uri(f"sqlite:///{APPCFG.sqldb_directory}")
execute_query = QuerySQLDataBaseTool(db=db)

# Crear la cadena de consultas SQL
write_query = create_sql_query_chain(APPCFG.langchain_llm, db)

# Plantilla para la respuesta
answer_prompt = PromptTemplate.from_template(APPCFG.agent_llm_system_role)
answer = answer_prompt | APPCFG.langchain_llm | StrOutputParser()

# Paso 1: Convertir lenguaje natural a SQL usando la plantilla
def convert_to_sql(state: QueryState):
    sql_query = write_query.invoke({"question": state["user_input"]}).strip()
    state["sql_query"] = sql_query
    state["query"] = state["sql_query"]  # Asignar el SQL generado a 'query'
    return state

# Paso 2: Ejecutar la consulta SQL
def execute_sql(state: QueryState):
    try:
        response = execute_query.invoke({"query": state["sql_query"]})
        state["result"] = str(response)
    except Exception as e:
        state["result"] = f"Error ejecutando SQL: {str(e)}"
    return state

# Paso 3: Generar la respuesta utilizando el LLM
def generate_answer(state: QueryState):
    try:
        # Verificar si no hay resultados en la base de datos
        if not state["result"].strip() or "no rows" in state["result"].lower():  # Ajuste para manejar respuestas vacías
            state["result"] = "No hay esa información disponible."
        else:
            response = answer.invoke({
                "query": state["query"],  # Pasar 'query'
                "question": state["user_input"],  # Pasar 'question'
                "result": state["result"]  # Pasar 'result'
            })
            state["result"] = str(response)
    except Exception as e:
        state["result"] = f"Error generando respuesta: {str(e)}"
    return state

# Construcción del flujo
graph = StateGraph(QueryState)
graph.add_node("convert_to_sql", convert_to_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("generate_answer", generate_answer)

graph.add_edge("convert_to_sql", "execute_sql")
graph.add_edge("execute_sql", "generate_answer")
graph.add_edge("generate_answer", END)

graph.set_entry_point("convert_to_sql")
app = graph.compile()

# Ejemplo de ejecución
if __name__ == "__main__":
    user_input = "zapatos"
    state = app.invoke({"user_input": user_input})
    print("Resultado final:", state["result"])


Resultado final: No hay esa información disponible.


In [39]:
if __name__ == "__main__":
    user_input = ("Donde nos ubicamos")
    state = app.invoke({"user_input": user_input})
    print("Resultado final:", state["result"])

Esquema de la base de datos: []
Relevancia de la consulta: False


KeyError: 'result'